# SPX Options Chain – Full Dashboard Logic EDA

This notebook replicates the core logic of your `dashboard.py` Streamlit app in a Jupyter environment.

You can:
- Load data from your Postgres/Neon database **or** from a local CSV.
- Reuse the same calculations: IV skew, OI & Volume, Gamma exposure, Term structure, OI change, spreads, vol surface, etc.
- Tweak parameters (spot, risk-free rate, expiration, strike range…) by editing Python variables instead of using Streamlit widgets.


## 1. Imports

In [20]:
import math
from datetime import date, datetime, timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"


from statsmodels.nonparametric.smoothers_lowess import lowess

# Optional: only needed if you load directly from Postgres/Neon
try:
    import psycopg2
except ImportError:
    psycopg2 = None
    print("psycopg2 not installed. Install it if you want to query the DB directly.")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

PX_TEMPLATE = "plotly_dark"


## 2. Configuration
Edit this cell to point to your DB or CSV, and to control the main analysis parameters.

In [21]:
# ---- Data source config ----
USE_DB = False          # True → query Postgres/Neon; False → load from CSV
CSV_PATH = "data/spx_chain.csv"  # used only if USE_DB = False

TABLE_LATEST = "spx_chain"   # latest snapshot table
TABLE_HIST   = "spx_chain"   # history table with run_ts

# DB credentials (edit with your real values, or load from env vars)
DB_HOST = "YOUR_DB_HOST"
DB_NAME = "YOUR_DB_NAME"
DB_USER = "YOUR_DB_USER"
DB_PASS = "YOUR_DB_PASS"
DB_SSLMODE = "require"   # or e.g. "prefer"

# ---- Global analysis parameters ----
# These will be auto-filled after loading df, but you can override them later.
SPOT = None          # If None, will be estimated from ATM strike
RISK_FREE = 0.03     # risk-free rate (annual)
SHOW_CALLS = True
SHOW_PUTS = True

# You can later set: EXPIRATION, STRIKE_LOW, STRIKE_HIGH explicitly once df is loaded.
EXPIRATION = None
STRIKE_LOW = None
STRIKE_HIGH = None


## 3. Helper Functions (DB + Math)

In [22]:
def get_conn():
    if not psycopg2:
        raise ImportError("psycopg2 is not available. Install it or set USE_DB = False.")
    return psycopg2.connect(
        host=DB_HOST,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
        sslmode=DB_SSLMODE,
        port=5432,
    )


def q(sql, params=None):
    """Run a SQL query and return a DataFrame."""
    if params is None:
        params = []
    with get_conn() as conn:
        return pd.read_sql(sql, conn, params=params)


def load_run_dates():
    """Return distinct run_ts from history table, newest first."""
    try:
        df_dates = q(f"SELECT DISTINCT run_ts FROM {TABLE_HIST} ORDER BY run_ts DESC")
        return list(df_dates["run_ts"])
    except Exception as e:
        print("Error loading run dates:", e)
        return []


def prev_run_date(dates, cur_dt):
    if not dates or cur_dt not in dates:
        return None
    idx = dates.index(cur_dt)
    return dates[idx + 1] if idx + 1 < len(dates) else None


def load_chain_by_run(run_ts):
    return q(
        f"""
        SELECT run_ts, expiration_date, strike, cp, last, bid, ask, volume, oi, iv
        FROM {TABLE_HIST}
        WHERE run_ts = %s
        ORDER BY expiration_date, strike, cp
        """, [run_ts]
    )


def load_chain_two_days(run_date_cur, run_date_prev):
    cur = load_chain_by_run(run_date_cur)
    prv = load_chain_by_run(run_date_prev) if run_date_prev else pd.DataFrame(columns=cur.columns)
    return cur, prv


# ---- Math helpers ----

def yearfrac(d0: date, d1: date) -> float:
    return max((d1 - d0).days, 0) / 365.0


def bs_d1(S, K, r, sigma, T):
    if S <= 0 or K <= 0 or sigma <= 0 or T <= 0:
        return np.nan
    return (np.log(S / K) + (r + 0.5 * sigma * sigma) * T) / (sigma * np.sqrt(T))


def bs_gamma(S, K, r, sigma, T):
    if S <= 0 or K <= 0 or sigma <= 0 or T <= 0:
        return 0.0
    d1 = bs_d1(S, K, r, sigma, T)
    pdf = 1.0 / np.sqrt(2 * np.pi) * np.exp(-0.5 * d1 * d1)
    return pdf / (S * sigma * np.sqrt(T))


def nearest_strike_iv(df, exp, spot):
    dfe = df[df["expiration_date"].eq(exp)]
    if dfe.empty:
        return np.nan
    idx = (dfe["strike"] - spot).abs().idxmin()
    k = dfe.loc[idx, "strike"]
    ivc = dfe[(dfe["strike"] == k) & (dfe["cp"] == "C")]["iv"].dropna().mean()
    ivp = dfe[(dfe["strike"] == k) & (dfe["cp"] == "P")]["iv"].dropna().mean()
    return np.nanmean([ivc, ivp])


## 4. Load Data

In [23]:
if USE_DB:
    run_dates = load_run_dates()
    print(f"Found {len(run_dates)} run_ts in history table.")
    if run_dates:
        print("First 5 run_ts (newest first):")
        for d in run_dates[:5]:
            print("  ", d)
    else:
        print("No history found; will fall back to latest table.")
else:
    run_dates = []

# Choose which run_ts to use when reading from history
# By default, use the most recent one (if exists)
if run_dates:
    RUN_TS = run_dates[0]
    print("Using RUN_TS =", RUN_TS)
else:
    RUN_TS = None

if USE_DB:
    if RUN_TS is not None:
        df = load_chain_by_run(RUN_TS)
        print("Loaded from history table.")
    else:
        df = q(f"SELECT run_ts, expiration_date, strike, cp, last, bid, ask, volume, oi, iv FROM {TABLE_LATEST}")
        print("Loaded from latest table.")
else:
    df = pd.read_csv(CSV_PATH)
    print("Loaded from CSV:", CSV_PATH)

print("Rows:", len(df))
df.head()


Loaded from CSV: data/spx_chain.csv
Rows: 79762


,run_ts,expiration_date,strike,cp,last,bid,ask,volume,oi,iv
0,2025-11-12 21:30:04.479823+00,2025-11-21,200.0,C,6534.75,6641.4,6659.50,0,85,0.0000
1,2025-11-12 21:30:04.479823+00,2025-11-21,200.0,P,0.03,0.0,0.05,0,2616,6.2206
2,2025-11-12 21:30:04.479823+00,2025-11-21,400.0,C,6300.50,6441.7,6459.80,0,61,0.0000
3,2025-11-12 21:30:04.479823+00,2025-11-21,400.0,P,0.03,0.0,0.05,0,3334,4.9520
4,2025-11-12 21:30:04.479823+00,2025-11-21,600.0,C,6068.18,6241.9,6260.00,0,16,0.0000


## 5. Basic Cleaning & Types

In [24]:
if "expiration_date" in df.columns:
    df["expiration_date"] = pd.to_datetime(df["expiration_date"]).dt.date

for c in ["strike", "last", "bid", "ask", "iv", "volume", "oi"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79762 entries, 0 to 79761
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   run_ts           79762 non-null  object 
 1   expiration_date  79762 non-null  object 
 2   strike           79762 non-null  float64
 3   cp               79762 non-null  object 
 4   last             79762 non-null  float64
 5   bid              79762 non-null  float64
 6   ask              79762 non-null  float64
 7   volume           79762 non-null  int64  
 8   oi               79762 non-null  int64  
 9   iv               79762 non-null  float64
dtypes: float64(5), int64(2), object(3)
memory usage: 6.1+ MB


## 6. Spot Detection & Core Filters

In [25]:
# --- Better SPOT detection from the options chain (ATM strike proxy) ---
if SPOT is None:
    if "volume" in df.columns and df["volume"].notna().any():
        try:
            atm_strike = float(df.loc[df["volume"].idxmax(), "strike"])
        except Exception:
            atm_strike = float(df["strike"].median(skipna=True))
    else:
        atm_strike = float(df["strike"].median(skipna=True))
    SPOT = atm_strike

print(f"Estimated SPOT = {SPOT:.2f}")
print(f"Risk-free rate (RISK_FREE) = {RISK_FREE:.4f}")

# Pick default expiration and strike range if not already set
expiries = sorted(df["expiration_date"].unique())
if EXPIRATION is None and expiries:
    EXPIRATION = expiries[0]

expiries, EXPIRATION


Estimated SPOT = 6860.00
Risk-free rate (RISK_FREE) = 0.0300


([datetime.date(2025, 11, 12),
  datetime.date(2025, 11, 13),
  datetime.date(2025, 11, 14),
  datetime.date(2025, 11, 17),
  datetime.date(2025, 11, 18),
  datetime.date(2025, 11, 19),
  datetime.date(2025, 11, 20),
  datetime.date(2025, 11, 21),
  datetime.date(2025, 11, 24),
  datetime.date(2025, 11, 25),
  datetime.date(2025, 11, 26),
  datetime.date(2025, 11, 28),
  datetime.date(2025, 12, 1),
  datetime.date(2025, 12, 2),
  datetime.date(2025, 12, 3),
  datetime.date(2025, 12, 4),
  datetime.date(2025, 12, 5),
  datetime.date(2025, 12, 8),
  datetime.date(2025, 12, 9),
  datetime.date(2025, 12, 10),
  datetime.date(2025, 12, 11),
  datetime.date(2025, 12, 12),
  datetime.date(2025, 12, 15),
  datetime.date(2025, 12, 16),
  datetime.date(2025, 12, 17),
  datetime.date(2025, 12, 18),
  datetime.date(2025, 12, 19),
  datetime.date(2025, 12, 22),
  datetime.date(2025, 12, 23),
  datetime.date(2025, 12, 26),
  datetime.date(2025, 12, 31),
  datetime.date(2026, 1, 2),
  datetime.date(2

In [26]:
# Strike range
dfe_all_exp = df[df["expiration_date"].eq(EXPIRATION)].copy()
min_k, max_k = float(dfe_all_exp["strike"].min()), float(dfe_all_exp["strike"].max())

if STRIKE_LOW is None:
    STRIKE_LOW = min_k
if STRIKE_HIGH is None:
    STRIKE_HIGH = max_k

print(f"Using expiration = {EXPIRATION}")
print(f"Strike range = [{STRIKE_LOW}, {STRIKE_HIGH}]")

mask = (
    ((df["cp"].eq("C") & SHOW_CALLS) | (df["cp"].eq("P") & SHOW_PUTS))
    & df["expiration_date"].eq(EXPIRATION)
)
dfe = df[mask].copy().sort_values("strike")
dfe = dfe[(dfe["strike"] >= STRIKE_LOW) & (dfe["strike"] <= STRIKE_HIGH)]

print("Filtered rows for selected expiration & strike range:", len(dfe))
dfe.head()


Using expiration = 2025-11-12
Strike range = [2800.0, 8800.0]
Filtered rows for selected expiration & strike range: 450


,run_ts,expiration_date,strike,cp,last,bid,ask,volume,oi,iv
9420,2025-11-12 21:30:04.479823+00,2025-11-12,2800.0,C,4060.00,4048.8,4053.90,1,0,0.0
9421,2025-11-12 21:30:04.479823+00,2025-11-12,2800.0,P,0.05,0.0,0.05,2,12,0.0
9422,2025-11-12 21:30:04.479823+00,2025-11-12,3000.0,C,0.00,3848.8,3855.30,0,0,0.0
9423,2025-11-12 21:30:04.479823+00,2025-11-12,3000.0,P,0.05,0.0,0.05,0,59,0.0
9424,2025-11-12 21:30:04.479823+00,2025-11-12,3200.0,C,0.00,3648.8,3655.30,0,0,0.0


## 7. IV Skew (Single Expiration)

In [27]:
fig_skew = px.scatter(
    dfe,
    x="strike",
    y="iv",
    color="cp",
    template=PX_TEMPLATE,
    title=f"IV vs Strike · {EXPIRATION}",
)
fig_skew.update_traces(marker=dict(size=6, opacity=0.9))
fig_skew.show()


## 8. OI & Volume (ATM Zoom + Percentile Capping)

In [28]:
use_log = False
pct_cap = 99
atm_zoom = True
band = 25  # +/- % around SPOT

df_zoom = dfe.copy()
if atm_zoom and np.isfinite(float(SPOT)):
    lo = float(SPOT) * (1 - band / 100.0)
    hi = float(SPOT) * (1 + band / 100.0)
    df_zoom = df_zoom[(df_zoom["strike"] >= lo) & (df_zoom["strike"] <= hi)]
    if df_zoom.empty:
        print("No strikes in ATM window; using full range instead.")
        df_zoom = dfe.copy()

df_zoom = df_zoom.sort_values(["expiration_date", "strike"])

def add_capped(series: pd.Series, q: int):
    if series.dropna().empty or q >= 100:
        return series
    hi_val = float(np.nanpercentile(series, q))
    return series.clip(upper=hi_val)

oi_df = df_zoom.groupby(["strike", "cp"], as_index=False)["oi"].sum()
vol_df = df_zoom.groupby(["strike", "cp"], as_index=False)["volume"].sum()

oi_df["oi_capped"] = add_capped(oi_df["oi"], pct_cap)
vol_df["vol_capped"] = add_capped(vol_df["volume"], pct_cap)

fig_oi = px.bar(
    oi_df,
    x="strike",
    y="oi_capped",
    color="cp",
    barmode="group",
    template=PX_TEMPLATE,
    title="Open Interest by Strike",
)
fig_oi.update_yaxes(type="log" if use_log else "linear", title="OI")
fig_oi.show()

fig_vol = px.bar(
    vol_df,
    x="strike",
    y="vol_capped",
    color="cp",
    barmode="group",
    template=PX_TEMPLATE,
    title="Volume by Strike (today)",
)
fig_vol.update_yaxes(type="log" if use_log else "linear", title="Volume")
fig_vol.show()


## 9. Dealer Gamma Exposure (Approximate)

In [29]:
CONTRACT_MULT = 100.0
today = date.today()

g = dfe.copy()
g["T"] = g["expiration_date"].apply(lambda d: yearfrac(today, d))
g["sigma"] = pd.to_numeric(g["iv"], errors="coerce")
g["gamma"] = g.apply(
    lambda r2: bs_gamma(SPOT, r2["strike"], RISK_FREE, r2["sigma"], r2["T"])
    if (r2["sigma"] and r2["sigma"] > 0 and r2["T"] > 0)
    else 0.0,
    axis=1,
)
g["gex"] = -g["gamma"] * g["oi"].fillna(0) * CONTRACT_MULT * (SPOT**2)

curve = g.groupby("strike", as_index=False)["gex"].sum().sort_values("strike")
curve["cum_gex"] = curve["gex"].cumsum()

# Flip level = zero-cross of cum_gex
flip_strike = None
sgn = np.sign(curve["cum_gex"])
change_idx = np.where(np.diff(sgn) != 0)[0]
if len(change_idx):
    i = change_idx[0]
    x0, y0 = curve.loc[i, ["strike", "cum_gex"]]
    x1, y1 = curve.loc[i + 1, ["strike", "cum_gex"]]
    if (y1 - y0) != 0:
        flip_strike = float(x0 - y0 * (x1 - x0) / (y1 - y0))
    else:
        flip_strike = float(curve.loc[i, "strike"])

fig_g = px.line(curve, x="strike", y="gex", template=PX_TEMPLATE, title="GEX by Strike")
fig_g.show()

fig_c = px.line(
    curve,
    x="strike",
    y="cum_gex",
    template=PX_TEMPLATE,
    title="Cumulative GEX (zero-cross ≈ flip)",
)
if flip_strike is not None:
    fig_c.add_vline(x=flip_strike, line_dash="dash", line_width=2)
    fig_c.add_annotation(
        x=flip_strike,
        y=curve["cum_gex"].min(),
        text=f"Flip ~ {flip_strike:.1f}",
        showarrow=False,
        yshift=20,
    )
fig_c.show()


## 10. ATM IV Term Structure

In [30]:
atm_rows = []
for e in sorted(df["expiration_date"].unique()):
    iv_atm = nearest_strike_iv(df, e, float(SPOT))
    if np.isfinite(iv_atm):
        atm_rows.append({"expiration_date": e, "atm_iv": iv_atm})

term = pd.DataFrame(atm_rows).sort_values("expiration_date")
term.head()


,expiration_date,atm_iv
0,2025-11-12,2.91025
1,2025-11-13,0.06420
2,2025-11-14,0.09915
3,2025-11-17,0.10715
4,2025-11-18,0.11790


In [31]:
fig_term = px.line(
    term,
    x="expiration_date",
    y="atm_iv",
    markers=True,
    template=PX_TEMPLATE,
    title="ATM IV Across Expirations",
)
fig_term.show()


## 11. Contracts Table (Filtered)

In [32]:
cols = ["expiration_date", "cp", "strike", "bid", "ask", "last", "iv", "volume", "oi"]
dfe[cols].sort_values(["cp", "strike"]).head(20)


,expiration_date,cp,strike,bid,ask,last,iv,volume,oi
9420,2025-11-12,C,2800.0,4048.8,4053.9,4060.00,0.0,1,0
9422,2025-11-12,C,3000.0,3848.8,3855.3,0.00,0.0,0,0
9424,2025-11-12,C,3200.0,3648.8,3655.3,0.00,0.0,0,0
9426,2025-11-12,C,3400.0,3448.8,3453.9,0.00,0.0,0,0
9428,2025-11-12,C,3600.0,3248.8,3253.9,0.00,0.0,0,0
9430,2025-11-12,C,3800.0,3048.8,3053.9,0.00,0.0,0,0
9432,2025-11-12,C,4000.0,2848.8,2853.9,0.00,0.0,0,0
9434,2025-11-12,C,4200.0,2648.8,2653.9,0.00,0.0,0,0
9436,2025-11-12,C,4400.0,2448.8,2453.9,2456.60,0.0,7,5
9438,2025-11-12,C,4600.0,2248.8,2253.9,2248.03,0.0,6,7


## 12. Skew Overlay (Multi-Expiry)

In [33]:
exps_all = sorted(df["expiration_date"].unique())

# Pick top 6 expirations by total volume as default
vol_by_exp = (
    df.groupby("expiration_date")["volume"].sum(min_count=1).sort_values(ascending=False)
)
top_exps = list(vol_by_exp.index[: min(6, len(vol_by_exp))])

# You can override this list manually if you want:
exps_sel = top_exps

print("Selected expirations for overlay:", exps_sel)

cp_filter = []
if SHOW_CALLS:
    cp_filter.append("C")
if SHOW_PUTS:
    cp_filter.append("P")

base = df[df["cp"].isin(cp_filter)].copy()
base = base[base["expiration_date"].isin(exps_sel)]
base = base.dropna(subset=["iv", "strike"]).copy()

# X-axis: you can switch to moneyness if you prefer
USE_MONEYNESS = False

if USE_MONEYNESS:
    base["xvar"] = base["strike"] / float(SPOT)
    x_title = "Moneyness (K/S)"
else:
    base["xvar"] = base["strike"].astype(float)
    x_title = "Strike"

AVG_CP = True       # True → average calls & puts together
SMOOTH_MODE = "raw"  # "raw" or "smooth"
LOWESS_FRAC = 0.15

if AVG_CP:
    group_cols = ["expiration_date", "xvar"]
else:
    group_cols = ["expiration_date", "cp", "xvar"]

iv_grid = (
    base.groupby(group_cols, as_index=False)["iv"]
    .mean()
    .dropna(subset=["iv", "xvar"])
    .sort_values(group_cols)
)

def smooth_series(df_part):
    df_part = df_part.dropna(subset=["xvar", "iv"]).sort_values("xvar")
    if len(df_part) < 5:
        return df_part.rename(columns={"iv": "iv_smooth"})[["xvar", "iv_smooth"]]
    sm = lowess(df_part["iv"].values, df_part["xvar"].values, frac=LOWESS_FRAC, return_sorted=True)
    return pd.DataFrame({"xvar": sm[:, 0], "iv_smooth": sm[:, 1]})

if SMOOTH_MODE == "raw":
    if AVG_CP:
        fig_overlay = px.line(
            iv_grid,
            x="xvar",
            y="iv",
            color="expiration_date",
            template=PX_TEMPLATE,
            title="IV Skew Overlay (Raw)",
        )
    else:
        fig_overlay = px.line(
            iv_grid,
            x="xvar",
            y="iv",
            color="expiration_date",
            line_dash="cp",
            template=PX_TEMPLATE,
            title="IV Skew Overlay (Raw, Calls vs Puts)",
        )
else:
    smooth_curves = []
    if AVG_CP:
        for expi, g_ in iv_grid.groupby("expiration_date"):
            sm = smooth_series(g_)
            sm["expiration_date"] = expi
            smooth_curves.append(sm)
    else:
        for (expi, side), g_ in iv_grid.groupby(["expiration_date", "cp"]):
            sm = smooth_series(g_)
            sm["expiration_date"] = expi
            sm["cp"] = side
            smooth_curves.append(sm)
    smooth_df = pd.concat(smooth_curves, ignore_index=True) if smooth_curves else pd.DataFrame(
        columns=["xvar", "iv_smooth"]
    )

    if AVG_CP:
        fig_overlay = px.line(
            smooth_df,
            x="xvar",
            y="iv_smooth",
            color="expiration_date",
            template=PX_TEMPLATE,
            title="IV Skew Overlay (Smoothed)",
        )
    else:
        fig_overlay = px.line(
            smooth_df,
            x="xvar",
            y="iv_smooth",
            color="expiration_date",
            line_dash="cp",
            template=PX_TEMPLATE,
            title="IV Skew Overlay (Smoothed, Calls vs Puts)",
        )

fig_overlay.update_layout(xaxis_title=x_title, yaxis_title="Implied Volatility", hovermode="x unified")
fig_overlay.show()


Selected expirations for overlay: [datetime.date(2025, 11, 14), datetime.date(2025, 11, 13), datetime.date(2025, 11, 12), datetime.date(2025, 11, 17), datetime.date(2025, 11, 21), datetime.date(2025, 12, 19)]


## 13. OI Change (Day-over-Day Flows)

In [34]:
if USE_DB and run_dates:
    date_cur = RUN_TS
    date_prev = prev_run_date(run_dates, date_cur)
    print("Current run_ts:", date_cur)
    print("Previous run_ts:", date_prev)

    cur, prv = load_chain_two_days(date_cur, date_prev)
    if cur.empty:
        print("No rows for selected date.")
    else:
        exp_sel = sorted(cur["expiration_date"].unique())[0]
        print("Using expiration:", exp_sel)

        key_cols = ["expiration_date", "strike", "cp"]
        cur_e = cur[cur["expiration_date"].eq(exp_sel)][
            key_cols + ["oi", "volume", "iv"]
        ].rename(columns={"oi": "oi_cur", "volume": "vol_cur", "iv": "iv_cur"})
        prv_e = prv[prv["expiration_date"].eq(exp_sel)][key_cols + ["oi"]].rename(
            columns={"oi": "oi_prev"}
        )

        merged = pd.merge(cur_e, prv_e, on=key_cols, how="left")
        for c in ["oi_prev", "oi_cur", "vol_cur", "iv_cur"]:
            merged[c] = pd.to_numeric(merged[c], errors="coerce").fillna(0)
        merged["oi_change"] = merged["oi_cur"] - merged["oi_prev"]

        hm = merged.groupby(["strike", "cp"], as_index=False)["oi_change"].sum()
        fig_hm = px.bar(
            hm,
            x="strike",
            y="oi_change",
            color="cp",
            barmode="group",
            template=PX_TEMPLATE,
            title=f"OI Change by Strike — {date_prev or '?'} → {date_cur}",
        )
        fig_hm.show()

        topN = 50
        movers = merged.reindex(merged["oi_change"].abs().sort_values(ascending=False).index).head(topN)
        movers[["expiration_date", "cp", "strike", "oi_prev", "oi_cur", "oi_change", "vol_cur", "iv_cur"]].head(20)
else:
    print("OI change analysis requires DB with at least two run_ts in history.")


OI change analysis requires DB with at least two run_ts in history.


## 14. Net Positioning Tilt (Call OI − Put OI)

In [35]:
cur_all = df.copy()
exp_tilt = EXPIRATION  # you can change this
cur_e = cur_all[cur_all["expiration_date"].eq(exp_tilt)]

calls = cur_e[cur_e["cp"].eq("C")].groupby("strike", as_index=False)["oi"].sum().rename(
    columns={"oi": "call_oi"}
)
puts = cur_e[cur_e["cp"].eq("P")].groupby("strike", as_index=False)["oi"].sum().rename(
    columns={"oi": "put_oi"}
)
tilt = pd.merge(calls, puts, on="strike", how="outer").fillna(0.0)
tilt["tilt"] = tilt["call_oi"] - tilt["put_oi"]
tilt = tilt.sort_values("strike")

fig_tilt = px.bar(
    tilt,
    x="strike",
    y="tilt",
    template=PX_TEMPLATE,
    title=f"Net Tilt by Strike — {exp_tilt}",
)
fig_tilt.add_hline(y=0, line_dash="dash")
fig_tilt.show()

tilt.tail(20)


,strike,call_oi,put_oi,tilt
205,7100.0,383,0,383
206,7125.0,223,0,223
207,7150.0,246,0,246
208,7175.0,128,0,128
209,7200.0,381,0,381
210,7225.0,448,0,448
211,7250.0,71,0,71
212,7275.0,5,0,5
213,7300.0,200,0,200
214,7325.0,0,0,0


## 15. Spread Detector (Heuristic)

In [36]:
if USE_DB and run_dates:
    date_cur = RUN_TS
    date_prev = prev_run_date(run_dates, date_cur)
    cur, prv = load_chain_two_days(date_cur, date_prev)
    if cur.empty:
        print("No rows for selected date.")
    else:
        exp_sel = sorted(cur["expiration_date"].unique())[0]
        key = ["expiration_date", "strike", "cp"]

        cur_e = cur[cur["expiration_date"].eq(exp_sel)][key + ["oi"]].rename(columns={"oi": "oi_cur"})
        prv_e = prv[prv["expiration_date"].eq(exp_sel)][key + ["oi"]].rename(columns={"oi": "oi_prev"})
        m = pd.merge(cur_e, prv_e, on=key, how="left").fillna({"oi_prev": 0})
        m["oi_change"] = pd.to_numeric(m["oi_cur"], errors="coerce").fillna(0) - pd.to_numeric(
            m["oi_prev"], errors="coerce"
        ).fillna(0)

        min_abs = 100
        m = m[m["oi_change"].abs() >= min_abs]

        verts = []
        for cp_side, g_ in m.groupby("cp"):
            g2 = g_.sort_values("strike").reset_index(drop=True)
            for i in range(len(g2) - 1):
                r1, r2 = g2.iloc[i], g2.iloc[i + 1]
                ratio = abs(abs(r1["oi_change"]) - abs(r2["oi_change"])) / max(abs(r1["oi_change"]), 1)
                if ratio <= 0.2:
                    verts.append(
                        {
                            "type": "Vertical",
                            "cp": cp_side,
                            "k1": r1["strike"],
                            "k2": r2["strike"],
                            "oi_change_1": int(r1["oi_change"]),
                            "oi_change_2": int(r2["oi_change"]),
                        }
                    )
        verts_df = pd.DataFrame(verts)

        condors = []
        if not verts_df.empty:
            calls_v = verts_df[verts_df["cp"] == "C"].copy()
            puts_v = verts_df[verts_df["cp"] == "P"].copy()
            for _, c_ in calls_v.iterrows():
                avg_c = np.mean([abs(c_["oi_change_1"]), abs(c_["oi_change_2"])])
                best = None
                best_diff = 1e9
                for _, p_ in puts_v.iterrows():
                    avg_p = np.mean([abs(p_["oi_change_1"]), abs(p_["oi_change_2"])])
                    diff = abs(avg_c - avg_p)
                    if diff < best_diff:
                        best = p_
                        best_diff = diff
                if best is not None and best_diff / max(avg_c, 1) < 0.3:
                    condors.append(
                        {
                            "type": "IronCondor",
                            "call_k1": c_["k1"],
                            "call_k2": c_["k2"],
                            "put_k1": best["k1"],
                            "put_k2": best["k2"],
                            "avg_abs_oi_change": int(
                                (avg_c + np.mean([abs(best["oi_change_1"]), abs(best["oi_change_2"])])) / 2
                            ),
                        }
                    )
        condors_df = pd.DataFrame(condors).drop_duplicates()

        print("Vertical spread candidates:")
        display(verts_df.head(20))

        print("\nIron Condor candidates:")
        display(condors_df.head(20))
else:
    print("Spread detector requires DB with two run_ts in history.")


Spread detector requires DB with two run_ts in history.


## 16. Skew Overlay: Today vs Yesterday

In [37]:
if USE_DB and run_dates and len(run_dates) >= 2:
    date_cur = RUN_TS
    date_prev = prev_run_date(run_dates, date_cur)
    print("Current run_ts:", date_cur)
    print("Prev run_ts:", date_prev)

    if date_prev is None:
        print("No previous run found.")
    else:
        cur, prv = load_chain_two_days(date_cur, date_prev)
        if cur.empty or prv.empty:
            print("No data for comparison dates.")
        else:
            exp_opts = sorted(set(cur["expiration_date"].unique()).intersection(prv["expiration_date"].unique()))
            if not exp_opts:
                print("No common expirations to compare.")
            else:
                exp_cmp = exp_opts[0]
                print("Comparing expiration:", exp_cmp)

                USE_MONEYNESS = False
                AVG_CP = True
                USE_SMOOTHING = False
                LOWESS_FRAC_CMP = 0.18

                def prep(df0, label):
                    df1 = df0[df0["expiration_date"].eq(exp_cmp)].copy()
                    cp_sel = []
                    if SHOW_CALLS:
                        cp_sel.append("C")
                    if SHOW_PUTS:
                        cp_sel.append("P")
                    if cp_sel:
                        df1 = df1[df1["cp"].isin(cp_sel)]
                    if USE_MONEYNESS:
                        df1["xvar"] = df1["strike"].astype(float) / float(SPOT)
                        x_title = "Moneyness (K/S)"
                    else:
                        df1["xvar"] = df1["strike"].astype(float)
                        x_title = "Strike"
                    group_cols = ["xvar"] if AVG_CP else ["cp", "xvar"]
                    agg = df1.groupby(group_cols, as_index=False)["iv"].mean().dropna()
                    agg["run_label"] = label
                    return agg, x_title

                cur_g, x_title = prep(cur, f"{date_cur}")
                prv_g, _ = prep(prv, f"{date_prev}")
                all_g = pd.concat([cur_g, prv_g], ignore_index=True)

                def do_smooth(df_in, features):
                    out = []
                    if "cp" in features:
                        for (lab, side), g_ in df_in.groupby(["run_label", "cp"]):
                            g_ = g_.sort_values("xvar")
                            if len(g_) >= 5:
                                sm = lowess(
                                    g_["iv"].values, g_["xvar"].values, frac=LOWESS_FRAC_CMP, return_sorted=True
                                )
                                out.append(
                                    pd.DataFrame(
                                        {"xvar": sm[:, 0], "iv": sm[:, 1], "run_label": lab, "cp": side}
                                    )
                                )
                            else:
                                out.append(g_[["xvar", "iv", "run_label", "cp"]])
                    else:
                        for lab, g_ in df_in.groupby("run_label"):
                            g_ = g_.sort_values("xvar")
                            if len(g_) >= 5:
                                sm = lowess(
                                    g_["iv"].values, g_["xvar"].values, frac=LOWESS_FRAC_CMP, return_sorted=True
                                )
                                out.append(
                                    pd.DataFrame({"xvar": sm[:, 0], "iv": sm[:, 1], "run_label": lab})
                                )
                            else:
                                out.append(g_[["xvar", "iv", "run_label"]])
                    return pd.concat(out, ignore_index=True) if out else df_in

                plot_df = all_g.copy()
                if USE_SMOOTHING:
                    plot_df = do_smooth(plot_df, features=["cp"] if not AVG_CP else [])

                if AVG_CP:
                    fig_cmp = px.line(
                        plot_df,
                        x="xvar",
                        y="iv",
                        color="run_label",
                        template=PX_TEMPLATE,
                        title=f"IV Skew: {date_prev} vs {date_cur} — {exp_cmp}",
                    )
                else:
                    fig_cmp = px.line(
                        plot_df,
                        x="xvar",
                        y="iv",
                        color="run_label",
                        line_dash="cp",
                        template=PX_TEMPLATE,
                        title=f"IV Skew: {date_prev} vs {date_cur} — {exp_cmp} (Calls vs Puts)",
                    )
                fig_cmp.update_layout(xaxis_title=x_title, yaxis_title="Implied Volatility", hovermode="x unified")
                fig_cmp.show()
else:
    print("Skew comparison requires DB with at least two run_ts in history.")


Skew comparison requires DB with at least two run_ts in history.


## 17. 3D Volatility Surface (Moneyness × DTE × IV)

In [38]:
surf = df.copy()
if surf.empty:
    print("No data to render 3D surface.")
else:
    today = date.today()
    surf["dte"] = surf["expiration_date"].apply(lambda d: max((d - today).days, 0))
    surf["moneyness"] = surf["strike"] / float(SPOT)
    surf = surf.dropna(subset=["iv", "moneyness", "dte"])

    grid = (
        surf.groupby(["dte", "moneyness"], as_index=False)["iv"]
        .mean()
        .sort_values(["dte", "moneyness"])
    )
    piv = grid.pivot(index="dte", columns="moneyness", values="iv").sort_index()

    X = np.array(piv.columns)   # moneyness
    Y = np.array(piv.index)     # dte
    Z = piv.values              # iv

    fig_surf = go.Figure(data=[go.Surface(x=X, y=Y, z=Z, colorscale="Viridis")])
    fig_surf.update_layout(
        template=PX_TEMPLATE,
        title="IV Surface",
        scene=dict(
            xaxis_title="Moneyness (K/S)",
            yaxis_title="Days to Expiration",
            zaxis_title="Implied Volatility",
        ),
        height=700,
    )
    fig_surf.show()
